In [1]:
source("~/bin/lit_utils.R")
lib_text()
source("/rd1/user/lit/project/sORFs/sORFs.utils.R")

Warning message:
“程辑包‘stringr’是用R版本4.3.2 来建造的”


In [48]:
read.table("./m.z.K0.txt") -> m.z.K0
nrow(m.z.K0)
head(m.z.K0)
colnames(m.z.K0) <- c('m','z','K0')
m.z.K0$mz <- m.z.K0$m/m.z.K0$z
head(m.z.K0)

[1] 117437

,V1,V2,V3
,<dbl>,<int>,<dbl>
1,566.3352,2,0.7971748
2,582.3413,2,0.8018603
3,622.0287,1,0.9904973
4,922.0092,1,1.1935936
5,1221.9884,1,1.3878447
6,1221.9930,1,1.3647832


,m,z,K0,mz
,<dbl>,<int>,<dbl>,<dbl>
1,566.3352,2,0.7971748,283.1676
2,582.3413,2,0.8018603,291.1707
3,622.0287,1,0.9904973,622.0287
4,922.0092,1,1.1935936,922.0092
5,1221.9884,1,1.3878447,1221.9884
6,1221.9930,1,1.3647832,1221.9930


In [49]:
min(m.z.K0$z)

[1] 0

In [50]:
summary(m.z.K0)

       m                z               K0               mz         
 Min.   :   0.0   Min.   :0.000   Min.   :0.6268   Min.   :  65.84  
 1st Qu.: 588.7   1st Qu.:2.000   1st Qu.:0.8844   1st Qu.: 211.04  
 Median : 755.8   Median :3.000   Median :0.9959   Median : 289.34  
 Mean   : 767.6   Mean   :2.663   Mean   :1.0037   Mean   : 315.33  
 3rd Qu.: 941.5   3rd Qu.:3.000   3rd Qu.:1.1148   3rd Qu.: 395.45  
 Max.   :1701.8   Max.   :5.000   Max.   :1.5451   Max.   :1400.17  
                                                   NA's   :1974     

In [13]:
# 有1974个前体离子的mass和z都是0
filter(m.z.K0,m==0) %>% nrow()
filter(m.z.K0,z==0) %>% nrow()
filter(m.z.K0,m!=0) %>% filter(.,z==0) %>% nrow()

[1] 1974

[1] 1974

[1] 0

BEGIN IONS

TITLE=CAD20250514licq_BSEP_DDA_60min_21pcw_1_C8_T_T_Slot2-3_1_7020.316.316.0

CHARGE=0+

但是在peaks studio中查看precusor id是316的ms/ms，z并不为0

# 查找m差不多，但是K0不同的记录

In [36]:
m.z.K0 %>%
group_by(m) %>%
filter(n() > 1 & any(K0 != first(K0))) -> mz_same_k0_inconsistent

In [37]:
mz_same_k0_inconsistent %>% filter(!is.na(mz)) %>% head(10)

m,z,K0,mz
<dbl>,<int>,<dbl>,<dbl>
361.6901,5,0.7387166,72.33802
361.6901,2,0.7370920,180.84504
388.7209,5,0.7465342,77.74417
388.7209,2,0.7503218,194.36044
440.4795,4,0.7883276,110.11988
440.4795,4,0.8026262,110.11988


In [32]:
# 使用dplyr包中的filter和lag函数查找m相差小于0.02但K0不同的行
m.z.K0 %>%
  arrange(m) %>% # 按mz排序
  filter(abs(m - lag(m)) <= 0.02 & K0 != lag(K0) | # mz相差0.02且K0不同
         abs(m - lead(m)) <= 0.02 & K0 != lead(K0)) -> mz_basicly_same_k0_inconsistent

In [33]:
nrow(mz_basicly_same_k0_inconsistent)
distinct(mz_basicly_same_k0_inconsistent,m,z,K0,mz) %>% nrow()

[1] 114515

[1] 114515

In [34]:
# 使用dplyr包中的filter和lag函数查找m相差小于0.02但K0不同的行
m.z.K0 %>%
  arrange(m) %>% # 按mz排序
  filter(abs(m - lag(m)) <= 0.02 & abs(K0 - lag(K0)) >=0.1 | # mz相差0.02且K0不同
         abs(m - lead(m)) <= 0.02 & abs(K0 - lead(K0)) >=0.1) -> mz_basicly_same_k0_inconsistent_large

In [35]:
nrow(mz_basicly_same_k0_inconsistent_large)

[1] 18400

# Peaks output

In [44]:
read.table("./peaks_output_mgf/m.z.K0.txt") -> m.z.K0
colnames(m.z.K0) <- c('m','z','K0')
m.z.K0$mz <- m.z.K0$m/m.z.K0$z
summary(m.z.K0)
# 有1974个前体离子的mass和z都是0
filter(m.z.K0,m==0) %>% nrow()
filter(m.z.K0,z==0) %>% nrow()
filter(m.z.K0,m!=0) %>% filter(.,z==0) %>% nrow()

       m                z               K0               mz         
 Min.   : 228.1   Min.   :1.000   Min.   :0.6285   Min.   :  63.09  
 1st Qu.: 589.8   1st Qu.:2.000   1st Qu.:0.8839   1st Qu.: 212.44  
 Median : 749.9   Median :3.000   Median :0.9919   Median : 289.16  
 Mean   : 769.7   Mean   :2.681   Mean   :1.0006   Mean   : 315.93  
 3rd Qu.: 929.1   3rd Qu.:3.000   3rd Qu.:1.1074   3rd Qu.: 393.46  
 Max.   :1701.8   Max.   :6.000   Max.   :1.5459   Max.   :1446.65  

[1] 0

[1] 0

[1] 0

In [45]:
head(m.z.K0)

,m,z,K0,mz
,<dbl>,<int>,<dbl>,<dbl>
1,566.3333,2,0.7978323,283.1667
2,581.8443,2,0.8034255,290.9221
3,922.0063,1,1.1949136,922.0063
4,622.0286,1,0.9919432,622.0286
5,1221.9827,1,1.3889880,1221.9827
6,754.3099,2,0.8950251,377.1549
